# Customer Segmentation Platform — Feature Engineering

This notebook converts the raw customer dataset into meaningful behavioral features for customer segmentation.

The main engineered features are:

- Recency
- Frequency
- Monetary Value
- Average Order Value
- Engagement Score
- Discount Dependency
- Return Rate
- Online Purchase Ratio
- In-Store Purchase Ratio
- Average Items per Transaction
- Support Interaction Rate

The resulting dataset will be saved as:

`../outputs/clustering_features.csv`


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries loaded successfully")


## 2. Load the Raw Customer Dataset

In [ ]:
df = pd.read_csv("../data/customer_data.csv")

print("Dataset loaded successfully")
print("Shape:", df.shape)

df.head()


## 3. Inspect Dataset Columns

In [ ]:
print("Number of columns:", len(df.columns))
print("\nColumns:")

for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")


## 4. Check Required Columns

These are the raw columns required for the engineered customer features.


In [ ]:
required_columns = [
    "customer_id",
    "membership_years",
    "total_transactions",
    "total_sales",
    "days_since_last_purchase",
    "website_visits",
    "app_usage",
    "social_media_engagement",
    "avg_discount_used",
    "total_returned_items",
    "total_items_purchased",
    "online_purchases",
    "in_store_purchases",
    "avg_items_per_transaction",
    "customer_support_calls"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    print("Missing required columns:")
    for col in missing_columns:
        print("-", col)
else:
    print("All required columns are available.")


# 5. Recency

Recency measures how recently a customer purchased.

Lower values indicate more recent activity.


In [ ]:
df["Recency"] = df["days_since_last_purchase"]

print(df["Recency"].describe())


# 6. Frequency

Frequency measures transaction activity relative to membership duration.

Formula:

`Frequency = Total Transactions / Membership Years`

A minimum denominator of 0.1 prevents division by zero.


In [ ]:
df["Frequency"] = (
    df["total_transactions"] /
    df["membership_years"].clip(lower=0.1)
)

print(df["Frequency"].describe())


# 7. Monetary Value

Monetary Value represents the customer's total spending.


In [ ]:
df["Monetary_Value"] = df["total_sales"]

print(df["Monetary_Value"].describe())


# 8. Average Order Value

Average Order Value represents average spending per transaction.


In [ ]:
df["Average_Order_Value"] = (
    df["total_sales"] /
    df["total_transactions"].replace(0, np.nan)
)

df["Average_Order_Value"] = (
    df["Average_Order_Value"].fillna(0)
)

print(df["Average_Order_Value"].describe())


# 9. Engagement Score

The exact specification fields such as Email Interaction and Wishlist Activity are not present under those names in this dataset.

Therefore, the available behavioral activity variables are combined:

`Website Visits + App Usage + Social Media Engagement`


In [ ]:
df["Engagement_Score"] = (
    df["website_visits"] +
    df["app_usage"] +
    df["social_media_engagement"]
)

print(df["Engagement_Score"].describe())


# 10. Discount Dependency

The specification defines Discount Dependency as discounted orders divided by total orders.

The current dataset does not provide a column named `discounted_orders`. It does provide `avg_discount_used`, so that field is used as a discount-intensity proxy.

This is a proxy and is not mathematically identical to discounted-orders / total-orders.


In [ ]:
df["Discount_Dependency"] = (
    df["avg_discount_used"]
)

print(df["Discount_Dependency"].describe())


# 11. Return Rate

Return Rate measures the proportion of purchased items that were returned.

`Return Rate = Returned Items / Purchased Items`


In [ ]:
df["Return_Rate"] = (
    df["total_returned_items"] /
    df["total_items_purchased"].replace(0, np.nan)
).fillna(0)

print(df["Return_Rate"].describe())


# 12. Online Purchase Ratio

In [ ]:
df["Online_Purchase_Ratio"] = (
    df["online_purchases"] /
    df["total_transactions"].replace(0, np.nan)
).fillna(0)

print(df["Online_Purchase_Ratio"].describe())


# 13. In-Store Purchase Ratio

In [ ]:
df["InStore_Purchase_Ratio"] = (
    df["in_store_purchases"] /
    df["total_transactions"].replace(0, np.nan)
).fillna(0)

print(df["InStore_Purchase_Ratio"].describe())


# 14. Average Items per Transaction

In [ ]:
df["Avg_Items_Per_Transaction"] = (
    df["avg_items_per_transaction"]
)

print(df["Avg_Items_Per_Transaction"].describe())


# 15. Customer Support Interaction Rate

This measures support interactions relative to transaction activity.


In [ ]:
df["Support_Interaction_Rate"] = (
    df["customer_support_calls"] /
    df["total_transactions"].replace(0, np.nan)
).fillna(0)

print(df["Support_Interaction_Rate"].describe())


# 16. Review Engineered Features

In [ ]:
engineered_features = [
    "Recency",
    "Frequency",
    "Monetary_Value",
    "Average_Order_Value",
    "Engagement_Score",
    "Discount_Dependency",
    "Return_Rate",
    "Online_Purchase_Ratio",
    "InStore_Purchase_Ratio",
    "Avg_Items_Per_Transaction",
    "Support_Interaction_Rate"
]

print("Engineered features:")
for feature in engineered_features:
    print("-", feature)


# 17. Create Clustering Feature List

Only meaningful behavioral/customer features are selected for the first clustering model.

Customer IDs are retained separately for identification but are NOT used as clustering features.


In [ ]:
clustering_features = [
    "Recency",
    "Frequency",
    "Monetary_Value",
    "Average_Order_Value",
    "Engagement_Score",
    "Discount_Dependency",
    "Return_Rate",
    "Online_Purchase_Ratio",
    "InStore_Purchase_Ratio",
    "Avg_Items_Per_Transaction",
    "Support_Interaction_Rate"
]

print("Number of clustering features:", len(clustering_features))
print(clustering_features)


# 18. Create Modeling Dataset

In [ ]:
model_df = df[
    ["customer_id"] + clustering_features
].copy()

print("Model dataset shape:", model_df.shape)

model_df.head()


# 19. Check Missing Values

In [ ]:
missing_values = model_df[clustering_features].isnull().sum()

print("Missing values:")
print(missing_values)

print(
    "\nTotal missing values:",
    missing_values.sum()
)


# 20. Check Infinite Values

In [ ]:
infinite_values = np.isinf(
    model_df[clustering_features]
).sum()

print("Infinite values:")
print(infinite_values)

print(
    "\nTotal infinite values:",
    infinite_values.sum()
)


# 21. Replace Infinite Values

In [ ]:
model_df[clustering_features] = (
    model_df[clustering_features]
    .replace([np.inf, -np.inf], np.nan)
)

print("Infinite values replaced.")


# 22. Handle Missing Engineered Values

Any remaining missing engineered values are filled with zero.

This is appropriate here because the engineered ratio features use zero to represent the absence of the corresponding activity.


In [ ]:
model_df[clustering_features] = (
    model_df[clustering_features]
    .fillna(0)
)

print(
    "Remaining missing values:",
    model_df[clustering_features].isnull().sum().sum()
)


# 23. Feature Statistics

In [ ]:
model_df[clustering_features].describe().T


# 24. Check Feature Ranges

In [ ]:
feature_ranges = pd.DataFrame({
    "Minimum": model_df[clustering_features].min(),
    "Maximum": model_df[clustering_features].max(),
    "Mean": model_df[clustering_features].mean(),
    "Median": model_df[clustering_features].median(),
    "Std": model_df[clustering_features].std()
})

feature_ranges


# 25. Correlation Analysis

Highly correlated variables may contain redundant information.

The correlations will be reviewed before final model training.


In [ ]:
correlation = model_df[clustering_features].corr()

correlation


# 26. Strong Correlations

In [ ]:
corr_pairs = (
    correlation
    .where(np.triu(np.ones(correlation.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)

print("Strong positive correlations:")
print(corr_pairs[corr_pairs >= 0.70])

print("\nStrong negative correlations:")
print(corr_pairs[corr_pairs <= -0.70])


# 27. Check Duplicate Customers

In [ ]:
duplicate_customer_count = (
    model_df["customer_id"].duplicated().sum()
)

print(
    f"Duplicate customer IDs in modeling dataset: "
    f"{duplicate_customer_count:,}"
)


# 28. Final Modeling Dataset Preview

In [ ]:
model_df.head(10)


# 29. Final Feature List

In [ ]:
print("FINAL CLUSTERING FEATURES")
print("=" * 60)

for i, feature in enumerate(clustering_features, start=1):
    print(f"{i:02d}. {feature}")

print("=" * 60)
print(f"Total features: {len(clustering_features)}")
print(f"Customers: {len(model_df):,}")


# 30. Save Engineered Dataset

The customer ID is retained so that cluster assignments can later be mapped back to customers.


In [ ]:
output_path = "../outputs/clustering_features.csv"

model_df.to_csv(
    output_path,
    index=False
)

print("Saved successfully:")
print(output_path)


# 31. Verify Saved Dataset

In [ ]:
saved_df = pd.read_csv("../outputs/clustering_features.csv")

print("Saved dataset shape:", saved_df.shape)
print("Saved columns:")
print(list(saved_df.columns))

saved_df.head()


# Feature Engineering Summary

The raw customer dataset has now been transformed into a customer segmentation modeling dataset.

## Engineered features

- **Recency** — days since last purchase
- **Frequency** — transactions relative to membership duration
- **Monetary Value** — total customer sales
- **Average Order Value** — sales per transaction
- **Engagement Score** — website, app, and social activity
- **Discount Dependency** — discount-intensity proxy based on average discount used
- **Return Rate** — returned items relative to purchased items
- **Online Purchase Ratio**
- **In-Store Purchase Ratio**
- **Average Items per Transaction**
- **Support Interaction Rate**

## Output

The final modeling dataset is saved to:

`../outputs/clustering_features.csv`

## Next Step

Proceed to **03_Clustering.ipynb**.

The next stage will:

1. Load `clustering_features.csv`
2. Scale the clustering features
3. Test different numbers of clusters
4. Use the Elbow Method
5. Evaluate Silhouette Score
6. Train K-Means
7. Assign each customer a cluster
8. Save the cluster assignments
